# NB_04 -- Data Quality

Runs all data quality checks across Bronze and Silver layers.
Results are stored in gold_dq_issues for direct consumption in Power BI.

Checks implemented:

Referential integrity:
- orphan_order_details: order lines with no matching order header
- orphan_shipments: shipment records with no matching order header
- missing_customer: orders referencing a CustomerID not in Customers
- missing_product: order lines referencing a ProductID not in Products
- negative_quantity: order lines with Quantity <= 0

Business consistency:
- shipment_before_order: ShipmentDate earlier than OrderDate (legacy data issue)
- unitprice_discrepancy: sale price differs from current catalog price (expected -- SCD behavior)

Shipments redundancy cross-validation:
- shipment_customer_mismatch: CustomerID in Shipments differs from CustomerID in Orders
- shipment_product_mismatch: ProductID in Shipments differs from ProductID in Order_Details
- shipment_employee_mismatch: EmployeeID in Shipments differs from EmployeeID in Orders

Each issue is tagged with a severity level: Critical, High, Medium, or Info.


In [ ]:

from pyspark.sql.functions import *
from functools import reduce

print("NB_04 Data Quality started")

In [ ]:
# Load all required tables once to avoid repeated reads
df_ord  = spark.table("silver_orders")
df_od   = spark.table("silver_order_details")
df_ship = spark.table("silver_shipments")

ord_ids  = df_ord.select("OrderID")
cust_ids = spark.table("silver_customers").select("CustomerID")
prod_ids = spark.table("silver_products").select("ProductID")

checks = []

# -- REFERENTIAL INTEGRITY ---------------------------------------------------

# Order lines without a matching order header
orphan_od = (df_od.join(ord_ids, "OrderID", "left_anti")
    .select(lit("orphan_order_details").alias("check_name"),
            lit("Critical").alias("severity"),
            lit("silver_order_details").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            lit("OrderID in order_details has no matching record in orders").alias("description")))
checks.append(orphan_od)

# Shipment records without a matching order header
orphan_ship = (df_ship.join(ord_ids, "OrderID", "left_anti")
    .select(lit("orphan_shipments").alias("check_name"),
            lit("Critical").alias("severity"),
            lit("silver_shipments").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            lit("OrderID in shipments has no matching record in orders").alias("description")))
checks.append(orphan_ship)

# Orders with a CustomerID not found in Customers
missing_cust = (df_ord.join(cust_ids, "CustomerID", "left_anti")
    .select(lit("missing_customer").alias("check_name"),
            lit("High").alias("severity"),
            lit("silver_orders").alias("table_name"),
            col("CustomerID").cast("string").alias("key_value"),
            lit("CustomerID in orders not found in customers").alias("description")))
checks.append(missing_cust)

# Order lines with a ProductID not found in Products
missing_prod = (df_od.join(prod_ids, "ProductID", "left_anti")
    .select(lit("missing_product").alias("check_name"),
            lit("High").alias("severity"),
            lit("silver_order_details").alias("table_name"),
            col("ProductID").cast("string").alias("key_value"),
            lit("ProductID in order_details not found in products").alias("description")))
checks.append(missing_prod)

# Negative or zero quantity -- no valid sale can have quantity <= 0
neg_qty = (df_od.filter(col("Quantity") <= 0)
    .select(lit("negative_quantity").alias("check_name"),
            lit("High").alias("severity"),
            lit("silver_order_details").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            concat(lit("Quantity="), col("Quantity").cast("string")).alias("description")))
checks.append(neg_qty)

print("Referential integrity checks defined")


In [ ]:
# -- BUSINESS CONSISTENCY ---------------------------------------------------

# Shipment before order date
# Expected to fire for all rows because of the legacy data issue (shipments 2007-2012 vs orders 2016-2021)
# This check confirms and quantifies the known issue rather than flagging a surprise
ship_before = (df_ship.alias("s")
    .join(df_ord.select("OrderID","OrderDate").alias("o"), "OrderID", "inner")
    .filter(col("ShipmentDate") < col("OrderDate"))
    .select(lit("shipment_before_order").alias("check_name"),
            lit("High").alias("severity"),
            lit("silver_shipments").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            concat(
                lit("ShipmentDate="), col("ShipmentDate").cast("string"),
                lit(" is before OrderDate="), col("OrderDate").cast("string")
            ).alias("description")))
checks.append(ship_before)

# UnitPrice discrepancy: sale price vs current catalog price
# NOTE: this is EXPECTED behavior, not a bug.
# Order_Details.UnitPrice = historical price at time of sale (a degenerate fact)
# Products.UnitPrice = current catalog price
# Labeled as Info severity -- it documents SCD behavior for the presentation
df_prod_price = spark.table("silver_products").select("ProductID","UnitPrice")
price_check = (df_od.alias("od")
    .join(df_prod_price.alias("p"), "ProductID", "inner")
    .filter(col("od.UnitPrice") != col("p.UnitPrice"))
    .select(lit("unitprice_discrepancy").alias("check_name"),
            lit("Info").alias("severity"),
            lit("silver_order_details").alias("table_name"),
            col("od.OrderID").cast("string").alias("key_value"),
            concat(
                lit("Sale="), col("od.UnitPrice").cast("string"),
                lit(" Catalog="), col("p.UnitPrice").cast("string"),
                lit(" Delta="), round(col("od.UnitPrice") - col("p.UnitPrice"), 2).cast("string")
            ).alias("description")))
checks.append(price_check)

print("Business consistency checks defined")


In [ ]:
# -- SHIPMENTS REDUNDANCY CROSS-VALIDATION ----------------------------------
# The Shipments table contains CustomerID, ProductID, and EmployeeID
# which are already in Orders and Order_Details. This is redundant data.
# If values disagree, it means the shipment was processed with different data
# than the original order -- a serious data integrity concern.

df_cross = (df_ship
    .join(df_ord.select("OrderID", col("CustomerID").alias("ord_CustID"), col("EmployeeID").alias("ord_EmpID")), "OrderID", "left")
    .join(df_od.select("OrderID","LineNo", col("ProductID").alias("od_ProdID")), ["OrderID","LineNo"], "left")
)

# CustomerID in shipment differs from order
cust_mismatch = (df_cross
    .filter(col("CustomerID").isNotNull() & col("ord_CustID").isNotNull() & (col("CustomerID") != col("ord_CustID")))
    .select(lit("shipment_customer_mismatch").alias("check_name"),
            lit("Critical").alias("severity"),
            lit("silver_shipments").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            concat(lit("Ship.CustomerID="), col("CustomerID").cast("string"),
                   lit(" vs Order.CustomerID="), col("ord_CustID").cast("string")).alias("description")))
checks.append(cust_mismatch)

# ProductID in shipment differs from order line
prod_mismatch = (df_cross
    .filter(col("ProductID").isNotNull() & col("od_ProdID").isNotNull() & (col("ProductID") != col("od_ProdID")))
    .select(lit("shipment_product_mismatch").alias("check_name"),
            lit("Critical").alias("severity"),
            lit("silver_shipments").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            concat(lit("Ship.ProductID="), col("ProductID").cast("string"),
                   lit(" vs OD.ProductID="), col("od_ProdID").cast("string")).alias("description")))
checks.append(prod_mismatch)

# EmployeeID in shipment differs from order
emp_mismatch = (df_cross
    .filter(col("EmployeeID").isNotNull() & col("ord_EmpID").isNotNull() & (col("EmployeeID") != col("ord_EmpID")))
    .select(lit("shipment_employee_mismatch").alias("check_name"),
            lit("Medium").alias("severity"),
            lit("silver_shipments").alias("table_name"),
            col("OrderID").cast("string").alias("key_value"),
            concat(lit("Ship.EmployeeID="), col("EmployeeID").cast("string"),
                   lit(" vs Order.EmployeeID="), col("ord_EmpID").cast("string")).alias("description")))
checks.append(emp_mismatch)

print("Cross-validation checks defined")


In [ ]:
# Union all checks and write to gold_dq_issues
df_dq = reduce(lambda a, b: a.union(b), checks)
df_dq = df_dq.withColumn("_checked_at", current_timestamp())

df_dq.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_dq_issues")

print(f"Total issues found: {df_dq.count()}")


In [ ]:
# Summary by severity and check type
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

spark.table("gold_dq_issues") \
    .groupBy("severity","check_name","table_name") \
    .count() \
    .orderBy(
        when(col("severity")=="Critical",1)
        .when(col("severity")=="High",2)
        .when(col("severity")=="Medium",3)
        .otherwise(4),
        col("count").desc()
    ).show(30, truncate=False)


In [ ]:
import pyspark.sql.functions as F

# Load the central data quality audit log table
df_dq = spark.table("gold_dq_issues")

print("=" * 80)
print("ADVANCED DATA QUALITY AUDIT REPORT")
print("=" * 80)

# SECTION 1: CRITICAL ERRORS (Referential Integrity & Cross-Validation Mismatches)
# These represent breaking data pipeline bugs that must be zero in production.
print("\n1. CRITICAL ERRORS AUDIT")
print("-" * 80)

df_critical = df_dq.filter(F.col("severity") == "Critical")
critical_count = df_critical.count()

print(f"Total critical rows found: {critical_count}")

if critical_count > 0:
    print("\nSummary of critical errors by type:")
    df_critical.groupBy("check_name", "table_name").count().show(truncate=False)
    
    print("\nData sample of critical errors (Top 10):")
    df_critical.select("check_name", "table_name", "key_value", "description").show(10, truncate=False)
else:
    print("Verification complete. No structural critical errors found in the system.")

# SECTION 2: UNEXPECTED ANOMALIES (Isolating the known legacy baseline data issues)
# Excludes the timeline time-travel and catalog slowly changing dimension variance.
print("\n" + "=" * 80)
print("2. UNEXPECTED HIGH/MEDIUM ANOMALIES")
print("-" * 80)

df_anomalies = df_dq.filter(
    (F.col("severity").isin("High", "Medium")) & 
    (~F.col("check_name").isin("shipment_before_order", "unitprice_discrepancy"))
)
anomalies_count = df_anomalies.count()

print(f"Total unexpected anomalies found: {anomalies_count}")

if anomalies_count > 0:
    print("\nSummary of hidden anomalies by type:")
    df_anomalies.groupBy("severity", "check_name", "table_name").count().show(truncate=False)
    
    print("\nData sample of hidden anomalies (Top 10):")
    df_anomalies.select("severity", "check_name", "key_value", "description").show(10, truncate=False)
else:
    print("Verification complete. All remaining high volume alerts originate exclusively from known legacy timeline shifts and historical price updates.")